In [4]:
%pip install python-pptx

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 27.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import collections 
import collections.abc
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from pptx.dml.color import RGBColor

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)
blank_layout = prs.slide_layouts[6]

# --- Color Palette ---
DARK = RGBColor(0x15, 0x16, 0x1C)
DARK2 = RGBColor(0x1D, 0x1F, 0x29)
ORANGE = RGBColor(0xE8, 0x62, 0x2C)
ORANGE_DEEP = RGBColor(0xC1, 0x42, 0x1C)
RED = RGBColor(0xA9, 0x22, 0x2E)
TEAL = RGBColor(0x12, 0xA5, 0x94)
TEAL_DEEP = RGBColor(0x0C, 0x7A, 0x6F)
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
LIGHT_BG = RGBColor(0xFA, 0xFA, 0xFA)
CARD_BG = RGBColor(0xF5, 0xF4, 0xF2)
TEXT_DARK = RGBColor(0x1E, 0x1E, 0x24)
MUTED = RGBColor(0x6B, 0x6F, 0x7B)

# --- Helper Functions ---
def add_bg(slide, color):
    bg = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, prs.slide_width, prs.slide_height)
    bg.fill.solid()
    bg.fill.fore_color.rgb = color
    bg.line.fill.background()

def add_text(slide, text, left, top, width, height, font_size, color=TEXT_DARK, bold=False, italic=False, align=PP_ALIGN.LEFT, anchor=MSO_ANCHOR.TOP):
    txBox = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = txBox.text_frame
    tf.word_wrap = True
    tf.vertical_anchor = anchor
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(font_size)
    p.font.color.rgb = color
    p.font.bold = bold
    p.font.italic = italic
    p.font.name = "Calibri"
    p.alignment = align
    return txBox

def add_icon(slide, icon_text, left, top, width, height, font_size, color=None, align=PP_ALIGN.LEFT):
    txBox = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = txBox.text_frame
    p = tf.paragraphs[0]
    p.text = icon_text
    p.font.size = Pt(font_size)
    # Ép sử dụng font Segoe UI Emoji để Windows render chuẩn
    p.font.name = "Segoe UI Emoji" 
    if color:
        p.font.color.rgb = color
    p.alignment = align
    return txBox

def add_shape(slide, shape_type, left, top, width, height, fill_color, line_color=None):
    shape = slide.shapes.add_shape(shape_type, Inches(left), Inches(top), Inches(width), Inches(height))
    shape.fill.solid()
    shape.fill.fore_color.rgb = fill_color
    if line_color:
        shape.line.color.rgb = line_color
        shape.line.width = Pt(1)
    else:
        shape.line.fill.background()
    return shape

def title_block(slide, kicker, title, is_dark=False):
    k_color = ORANGE if is_dark else ORANGE_DEEP
    t_color = WHITE if is_dark else TEXT_DARK
    add_text(slide, kicker.upper(), 0.8, 0.5, 10, 0.4, 14, color=k_color, bold=True)
    add_text(slide, title, 0.8, 0.9, 11.5, 0.8, 30, color=t_color, bold=True)

def add_footer(slide, page, is_dark=False):
    f_color = MUTED if not is_dark else RGBColor(0xA7, 0xAB, 0xB8)
    add_text(slide, "Đề tài KH&CN cấp cơ sở  •  T2026-07-06  •  Trương Minh Hiển", 0.8, 7.0, 8, 0.3, 11, color=f_color)
    add_text(slide, f"Trang {page}/14", 11.5, 7.0, 1.0, 0.3, 11, color=f_color, align=PP_ALIGN.RIGHT)

# --- Slide Layout Templates ---
def slide_title():
    slide = prs.slides.add_slide(blank_layout)
    add_bg(slide, DARK)
    add_shape(slide, MSO_SHAPE.OVAL, 10.0, -1.0, 5, 5, ORANGE_DEEP)
    add_text(slide, "THUYẾT MINH ĐỀ TÀI KHOA HỌC VÀ CÔNG NGHỆ CẤP CƠ SỞ", 0.8, 1.2, 9.5, 0.4, 14, color=ORANGE, bold=True)
    add_icon(slide, "🔥", 0.8, 1.8, 0.8, 0.8, 36)
    add_text(slide, "NGHIÊN CỨU CÁC PHƯƠNG PHÁP HỌC MÁY ĐỂ PHÁT HIỆN SỰ CỐ CHÁY NỔ QUA CAMERA CCTV", 0.8, 2.8, 11.2, 2.0, 38, color=WHITE, bold=True)
    add_shape(slide, MSO_SHAPE.RECTANGLE, 0.8, 5.2, 11.2, 0.02, RGBColor(0x3A, 0x3D, 0x4A))
    
    info = [("MÃ SỐ", "T2026-07-06", 0.8), ("CHỦ NHIỆM", "Trương Minh Hiển", 3.2), 
            ("GV HƯỚNG DẪN", "TS. Lê Thị Mỹ Hạnh", 6.0), ("CƠ QUAN CHỦ TRÌ", "Trường ĐH Bách khoa - ĐHĐN", 8.8)]
    for k, v, x in info:
        add_text(slide, k, x, 5.5, 2.6, 0.3, 11, color=ORANGE, bold=True)
        add_text(slide, v, x, 5.9, 2.6, 0.6, 13, color=WHITE)
    return slide

def slide_list(kicker, title, items, is_dark=False):
    slide = prs.slides.add_slide(blank_layout)
    add_bg(slide, DARK if is_dark else LIGHT_BG)
    title_block(slide, kicker, title, is_dark)
    y = 2.0
    for emoji, head, sub in items:
        add_shape(slide, MSO_SHAPE.ROUNDED_RECTANGLE, 0.8, y, 11.7, 1.0, DARK2 if is_dark else WHITE, line_color=RGBColor(0xE4,0xE3,0xE0) if not is_dark else None)
        add_icon(slide, emoji, 1.0, y+0.2, 0.6, 0.6, 28)
        add_text(slide, head, 1.8, y+0.15, 10.5, 0.4, 16, color=WHITE if is_dark else TEXT_DARK, bold=True)
        add_text(slide, sub, 1.8, y+0.5, 10.5, 0.4, 13, color=RGBColor(0xD8,0xDA,0xE2) if is_dark else MUTED)
        y += 1.15
    return slide

def slide_split(kicker, title, left_title, left_text, right_title, right_text, bottom_note=""):
    slide = prs.slides.add_slide(blank_layout)
    add_bg(slide, LIGHT_BG)
    title_block(slide, kicker, title)
    
    add_shape(slide, MSO_SHAPE.ROUNDED_RECTANGLE, 0.8, 2.0, 5.6, 3.8, CARD_BG)
    add_text(slide, left_title, 1.1, 2.2, 5.0, 0.4, 18, color=TEAL_DEEP, bold=True)
    add_text(slide, left_text, 1.1, 2.8, 5.0, 2.8, 14, color=TEXT_DARK)
    
    add_shape(slide, MSO_SHAPE.ROUNDED_RECTANGLE, 6.7, 2.0, 5.6, 3.8, CARD_BG)
    add_text(slide, right_title, 7.0, 2.2, 5.0, 0.4, 18, color=ORANGE_DEEP, bold=True)
    add_text(slide, right_text, 7.0, 2.8, 5.0, 2.8, 14, color=TEXT_DARK)
    
    if bottom_note:
        add_shape(slide, MSO_SHAPE.ROUNDED_RECTANGLE, 0.8, 6.0, 11.5, 0.8, ORANGE)
        add_text(slide, bottom_note, 1.0, 6.15, 11.1, 0.5, 14, color=WHITE, bold=True, align=PP_ALIGN.CENTER)
    return slide

def slide_cards(kicker, title, cards_data):
    slide = prs.slides.add_slide(blank_layout)
    add_bg(slide, LIGHT_BG)
    title_block(slide, kicker, title)
    
    width = 11.7 / len(cards_data) - 0.3
    x = 0.8
    for emoji, title_text, desc, color in cards_data:
        add_shape(slide, MSO_SHAPE.ROUNDED_RECTANGLE, x, 2.0, width, 4.0, CARD_BG)
        add_icon(slide, emoji, x, 2.4, width, 0.6, 36, align=PP_ALIGN.CENTER)
        add_text(slide, title_text, x+0.2, 3.3, width-0.4, 0.8, 16, color=color, bold=True, align=PP_ALIGN.CENTER)
        add_text(slide, desc, x+0.2, 4.1, width-0.4, 1.7, 13, color=MUTED, align=PP_ALIGN.CENTER)
        x += width + 0.3
    return slide

# --- Building Slides ---

# 1. Title
slide_title()

# 2. Cấp thiết (Thay icon an toàn: ⚠, ➕, 💲, ⏳)
s2 = slide_cards("Tính cấp thiết", "Cháy nổ vẫn là mối đe dọa hiện hữu tại Việt Nam", [
    ("⚠", "3.060", "vụ cháy trên toàn quốc mỗi năm, diễn biến vô cùng phức tạp.", RED),
    ("➕", "90+", "người tử vong và hàng trăm người bị thương thương tâm.", ORANGE_DEEP),
    ("💲", "786 Tỷ", "đồng thiệt hại tài sản, ảnh hưởng lớn đến kinh tế - xã hội.", TEAL_DEEP),
    ("⏳", "Độ trễ cao", "Hệ thống báo cháy truyền thống phát hiện chậm, thiếu ngữ cảnh hình ảnh.", TEXT_DARK)
])
add_shape(s2, MSO_SHAPE.ROUNDED_RECTANGLE, 0.8, 6.2, 11.7, 0.7, DARK)
add_text(s2, "Giải pháp: Tận dụng camera CCTV RGB sẵn có + AI tinh gọn để cảnh báo cháy nổ SỚM, xử lý CỤC BỘ.", 1.0, 6.35, 11.3, 0.5, 14, color=WHITE, bold=True, align=PP_ALIGN.CENTER)
add_footer(s2, 2)

# 3. Mục tiêu (💡, 💻, 🔒, 📱)
s3 = slide_list("Mục tiêu nghiên cứu", "Mục tiêu tổng quát & Cụ thể của đề tài", [
    ("💡", "Phát triển phương pháp học sâu tinh gọn", "Xây dựng mô hình nhận diện khói/lửa trực tiếp qua camera RGB thông thường thay vì ảnh nhiệt đắt đỏ."),
    ("💻", "Triển khai trên thiết bị vi tính biên (Edge Devices)", "Tối ưu hóa để chạy mượt mà trên Raspberry Pi 4B/5 hoặc Rock 5B+ mà không cần GPU đắt tiền."),
    ("🔒", "Xử lý cục bộ & Đảm bảo quyền riêng tư", "Module đọc dữ liệu trực tiếp trong mạng WiFi nội bộ, xử lý 100% Local không đẩy video lên Cloud."),
    ("📱", "Ứng dụng di động cảnh báo thời gian thực", "Phát triển app theo dõi trạng thái đám cháy, gửi cảnh báo tức thì với độ trễ cực thấp.")
])
add_footer(s3, 3)

# 4. Đối tượng & Phạm vi
s4 = slide_split("Đối tượng & Phạm vi", "Giới hạn phạm vi nghiên cứu thực tiễn", 
    "🎯 Đối tượng nghiên cứu", 
    "• Các phương pháp học sâu phát hiện khói và lửa.\n\n• Cấu trúc Spatiotemporal Learning (học không-thời gian).\n\n• Nền tảng vi máy tính nhúng & thiết bị biên: Raspberry Pi 4B/5, Rock 5B+.\n\n• Dữ liệu luồng video chuẩn RGB qua RTSP.",
    "📏 Phạm vi nghiên cứu",
    "• Về môi trường: Tập trung sự cố cháy nổ trong nhà (hộ gia đình, chung cư mini, xưởng nhỏ).\n\n• Về công nghệ: Chỉ dùng camera quang học thông thường, không dùng camera ảnh nhiệt.\n\n• Về xử lý: Hoàn toàn cục bộ trong mạng nội bộ.\n\n• Về thời gian: Thực hiện trong 12 tháng (06/2026 - 05/2027)."
)
add_footer(s4, 4)

# 5. Tổng quan thế giới (💿, 🌐, 📉, ❗)
s5 = slide_list("Tổng quan nghiên cứu", "Thế giới: Deep Learning dẫn đầu xu hướng phát hiện cháy", [
    ("💿", "CNN truyền thống (YOLO, ResNet, VGG)", "Đã đạt mAP > 90% trên các tập dữ liệu tĩnh (D-Fire, FIRESENSE, FLAME)."),
    ("🌐", "Kiến trúc Không-Thời gian (Spatiotemporal)", "Sử dụng Vision Transformer, 3D-CNN để theo dõi động lực học của khói/lửa qua chuỗi video (2025-2026)."),
    ("📉", "Giảm thiểu báo động giả (False Alarm)", "Xu hướng Self-supervised Learning giúp giảm FPR < 5%, hỗ trợ realtime mạnh mẽ."),
    ("❗", "Khoảng trống hiện tại (Gaps)", "Ít mô hình tập trung phát hiện giai đoạn khởi phát (Early-stage). Phụ thuộc quá nhiều vào GPU/Cloud đắt đỏ.")
])
add_footer(s5, 5)

# 6. Tổng quan VN
s6 = slide_split("Tình hình trong nước", "Việt Nam: Dư địa lớn cho Edge AI phòng cháy",
    "Trạng thái công nghệ PCCC hiện tại",
    "• Chủ yếu vẫn dựa vào cảm biến khói, cảm biến nhiệt vật lý (cần tiếp xúc trực tiếp).\n\n• Một số đơn vị thí điểm dùng camera AI Thermal (ảnh nhiệt) hoặc hệ thống giám sát rừng quy mô lớn.\n\n• Các nghiên cứu tại ĐH chủ yếu dùng YOLO cho an ninh tĩnh, chưa có mô hình Spatiotemporal tinh gọn cho camera thường.",
    "Cơ hội & Sự khác biệt của đề tài",
    "• Chưa có đề tài cấp cơ sở/bộ nào trùng lặp trực tiếp về việc đưa mô hình nhận diện khói lửa tinh gọn xuống thiết bị Raspberry Pi.\n\n• Phù hợp với bối cảnh thực tiễn Việt Nam: Tận dụng hạ tầng camera có sẵn ở khu trọ, chung cư mini để giảm chi phí đầu tư.\n\n• Chủ động hoàn toàn về công nghệ lõi."
)
add_footer(s6, 6)

# 7. Cách tiếp cận (👁, ⚙, ⚡, 🛡)
s7 = slide_cards("Cách tiếp cận", "Tiếp cận đa chiều từ Thuật toán đến Phần cứng", [
    ("👁", "Giải quyết thực tiễn", "Chuyển từ khái niệm 'hậu kiểm' (post-event) của CCTV sang 'cảnh báo sớm' (early warning) thời gian thực.", TEAL_DEEP),
    ("⚙", "Thuật toán lõi", "Tiếp cận học Không-Thời gian (Spatiotemporal), nhận diện sự biến đổi hình dáng khói/lửa thay vì chỉ nhận diện vật thể tĩnh.", ORANGE_DEEP),
    ("⚡", "Tối ưu phần cứng", "Nén mô hình (Pruning), Lượng tử hóa (INT8/FP16) để chạy trơn tru trên thiết bị biên nhúng.", RED),
    ("🛡", "Bảo mật & Dữ liệu", "Giao thức RTSP xử lý 100% mạng nội bộ. Tránh rò rỉ dữ liệu cá nhân lên các máy chủ đám mây.", TEAL)
])
add_footer(s7, 7)

# 8. Phương pháp NC (📖, 📂, 🧪, 📊)
s8 = slide_list("Phương pháp nghiên cứu", "Phương pháp luận & Thực nghiệm khoa học", [
    ("📖", "Phương pháp lý thuyết", "Kế thừa và phân tích các công bố SOTA (2025-2026) về nhận diện video và nén mạng Nơ-ron."),
    ("📂", "Tổng hợp & Tăng cường dữ liệu", "Thu thập từ nguồn mở (D-Fire, BoWFire...), làm sạch và mô phỏng nhiễu, sương mù, thay đổi ánh sáng."),
    ("🧪", "Mô hình hóa & Thực nghiệm", "Thiết lập kịch bản tiền xử lý, Fine-tune mô hình AI, Quantization và Deploy thực tế trên Rock 5B+/Pi 5."),
    ("📊", "Thống kê & Đối sánh", "Đo lường định lượng bằng mAP, F1-Score, FPS, Latency. So sánh trực tiếp với Baseline (YOLOv8, ViT).")
])
add_footer(s8, 8)

# 9. Kiến trúc linh hoạt
s9 = slide_split("Định hướng kiến trúc", "Hướng tiếp cận linh hoạt với Spatiotemporal Learning",
    "🧬 Tập trung vào Spatiotemporal (Không - Thời gian)",
    "• Đề tài KHÔNG cố định cứng ngắc vào một kiến trúc duy nhất.\n\n• Cốt lõi là học sự khuếch tán của khói và ngọn lửa qua các khung hình liên tiếp, chứ không chỉ học trên 1 bức ảnh tĩnh.\n\n• Sẽ thử nghiệm linh hoạt các biến thể của Vision Transformer (ViT), 3D-CNN hoặc Hybrid models để tìm ra mô hình nhẹ nhất.",
    "💡 Ví dụ tham khảo: V-JEPA 2",
    "• Kiến trúc V-JEPA 2 (Meta AI) là một tham chiếu lý thuyết tuyệt vời cho khả năng học tự giám sát (Self-supervised) động lực học từ video.\n\n• Đề tài có thể tùy biến các cơ chế của những mô hình tương tự để tối ưu hàm Loss, phạt nặng các trường hợp bỏ sót (False Negative) ở giai đoạn ngọn lửa mới bùng phát.",
    bottom_note="Mục tiêu tối thượng: Mô hình phải đủ NHẸ để chạy Edge, đủ SỚM để cảnh báo."
)
add_footer(s9, 9)

# 10. Tiến độ
s10 = slide_list("Nội dung & Tiến độ", "Kế hoạch triển khai trong 12 tháng (06/2026 - 05/2027)", [
    ("01", "Tháng 06 - 08/2026", "Chuẩn bị, tăng cường bộ dữ liệu. Thiết lập hạ tầng và lập trình module đọc luồng RTSP trong mạng LAN."),
    ("02", "Tháng 09 - 12/2026", "Cải tiến kiến trúc mô hình học Không - Thời gian. Tinh chỉnh (fine-tune) loss function cho early warning."),
    ("03", "Tháng 01 - 02/2027", "Tối ưu hóa mô hình (Quantization INT8/FP16) cho thiết bị biên. Xây dựng module tracking đám cháy."),
    ("04", "Tháng 03 - 05/2027", "Xây dựng Backend & Mobile App Flutter. Đánh giá thực nghiệm, viết bài báo khoa học và tổng kết.")
])
add_footer(s10, 10)

# 11. Sản phẩm (📋, 🤖, 📲)
s11 = slide_cards("Sản phẩm dự kiến", "Kết quả đầu ra của đề tài", [
    ("📋", "Khoa học & Đào tạo", "• 01 Bài báo quốc tế (Scopus/ISI Q1/Q2)\n• 01 Báo cáo tổng kết\n• Đào tạo chuyên sâu 02 Sinh viên (CV, Edge AI).", ORANGE_DEEP),
    ("🤖", "Thuật toán Edge AI", "• 01 Mô hình lượng tử hóa (< 50MB, FPS 15-25, F1 > 90%)\n• Module xử lý RTSP hoàn toàn cục bộ.", TEAL_DEEP),
    ("📲", "Hệ thống Thực tế", "• 01 Mobile App Flutter đa nền tảng (Push notification < 2s).\n• 01 Prototype phần cứng khép kín sẵn sàng deploy.", RED)
])
add_footer(s11, 11)

# 12. Chuyển giao
s12 = slide_split("Chuyển giao công nghệ", "Phương thức chuyển giao và Ứng dụng thực tiễn",
    "🤝 Phương thức chuyển giao",
    "• Công bố mã nguồn mở (MIT/Apache 2.0) giúp cộng đồng dễ dàng tiếp cận.\n\n• Cung cấp bộ Prototype phần cứng hoàn chỉnh cho các đơn vị thí điểm.\n\n• Hợp tác chuyên môn với NuverxAI và các đơn vị PCCC địa phương để tích hợp vào hệ thống có sẵn.",
    "🏢 Địa chỉ ứng dụng tiềm năng",
    "• Dân dụng: Hộ gia đình, nhà ở kết hợp kinh doanh, chung cư mini, ký túc xá, trường học.\n\n• Công nghiệp: Các cơ sở sản xuất vừa và nhỏ, nhà kho.\n\n• Địa bàn ưu tiên: Thí điểm tại các khu vực đông đúc ở Đà Nẵng (Liên Chiểu, Hải Châu)."
)
add_footer(s12, 12)

# 13. Tác động (🌟, 📈, 🎓)
s13 = slide_cards("Tác động & Lợi ích", "Giá trị mang lại cho Kinh tế, Xã hội và Khoa học", [
    ("🌟", "Kinh tế & Xã hội", "Bảo vệ tính mạng, tài sản người dân. Cung cấp giải pháp PCCC công nghệ cao với chi phí đầu tư và vận hành cực thấp.", ORANGE_DEEP),
    ("📈", "Khoa học & Công nghệ", "Tiên phong giải quyết bài toán Spatiotemporal trên thiết bị Edge. Cung cấp nguồn dữ liệu đối sánh giá trị cho cộng đồng AI.", TEAL_DEEP),
    ("🎓", "Giáo dục & Thương mại", "Nâng cao năng lực NCKH của sinh viên ĐH Bách khoa ĐHĐN. Mở ra cơ hội thương mại hóa thành sản phẩm Smart Home thực thụ.", RED)
])
add_footer(s13, 13)

# 14. End Slide
s14 = prs.slides.add_slide(blank_layout)
add_bg(s14, DARK)
add_shape(s14, MSO_SHAPE.OVAL, -1.0, 5.0, 6, 6, ORANGE_DEEP) 
add_shape(s14, MSO_SHAPE.OVAL, 8.0, -2.0, 6, 6, TEAL_DEEP)
add_text(s14, "XIN TRÂN TRỌNG CẢM ƠN", 1.0, 2.8, 11.3, 1.0, 54, color=WHITE, bold=True, align=PP_ALIGN.CENTER)
add_text(s14, "Hội đồng đã lắng nghe phần trình bày thuyết minh đề tài", 1.0, 4.0, 11.3, 1.0, 24, color=MUTED, align=PP_ALIGN.CENTER)

# Save
prs.save("NghienCuu_CameraCCTV_AI_FixedIcons.pptx")
print("Presentation with fixed icons generated successfully!")

Presentation with fixed icons generated successfully!
